<a href="https://colab.research.google.com/github/cemgurbey/sleep-log-data-analysis/blob/main/notebooks/sleep_log_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sleep Log Data Analysis

Standardized processing pipeline for longitudinal sleep diary + activity survey data (Morning and Evening diaries).
Reads raw survey workbooks, validates them, and produces publication-ready datasets for SPSS / R / Python:

- `AVERAGE_AND_SD_OUTPUTS.xlsx` — circular mean & SD of bedtime/waketime/in-bed/out-of-bed (weekday, weekend, total)
- `SL_OUTPUTS.xlsx` — survey completion counts per participant
- `SLx_OUTPUTS_ROW1..ROW4.xlsx` — daily longitudinal variables (sleep timing, activities, hourly media use, medications, POMS-A mood)
- `INVALID_DATETIME_ENTRIES_*.xlsx` + `ERROR_LOG.txt` — quality-assurance reports

**No data? No problem.** The pipeline ships with a synthetic data generator — run it as-is and you'll get example results immediately.

## 0. Install the package

In [ ]:
%pip install -q git+https://github.com/cemgurbey/sleep-log-data-analysis.git

In [ ]:
import sleeplog
print(f"sleeplog {sleeplog.__version__} ready")

## 1. Configuration

Study window and site (used for sunrise/sunset photoperiod via `astral`). Dates are plain `datetime.date` objects.

In [ ]:
from datetime import date
from sleeplog import SiteLocation, StudyConfig

config = StudyConfig(
    start_date=date(2022, 3, 22),
    end_date=date(2022, 3, 28),
    location=SiteLocation(
        name="Montreal",
        region="Canada",
        timezone="America/Montreal",
        latitude=45.5017,
        longitude=-73.5673,
    ),
)
print(f"Study window: {config.start_date} to {config.end_date} "
      f"({len(config.date_strings)} days)")

## 2. Data

By default, generate **artificial but realistic** Morning/Evening survey workbooks. Set `USE_SYNTHETIC = False` and upload your own `.xlsx` files instead (Filenames containing `morning` / `evening` are auto-detected).

In [ ]:
from pathlib import Path

USE_SYNTHETIC = True  # set to False to upload real survey files

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

if USE_SYNTHETIC:
    from sleeplog.synthetic import generate_survey_workbooks
    morning_path, evening_path = generate_survey_workbooks(
        data_dir, config, n_participants=12, seed=7
    )
else:
    try:
        from google.colab import files
        uploaded = files.upload()
        names = list(uploaded.keys())
        morning_path = next(n for n in names if "morning" in n.lower())
        evening_path = next(n for n in names if "evening" in n.lower())
    except ImportError:
        raise SystemExit("Not in Colab: place your files in ./data/ and set paths manually.")

print(f"Morning: {morning_path}")
print(f"Evening: {evening_path}")

## 3. Run the pipeline

In [ ]:
from sleeplog import run

written = run(config, morning_path, evening_path, out_dir="outputs")

## 4. Inspect the results

In [ ]:
import pandas as pd

out = Path("outputs")
avg_sd = pd.read_excel(out / "AVERAGE_AND_SD_OUTPUTS.xlsx")
completion = pd.read_excel(out / "SL_OUTPUTS.xlsx")

print("=== Survey completion (SL_OUTPUTS) ===")
display(completion.head())
print("\n=== Circular sleep statistics (AVERAGE_AND_SD_OUTPUTS, first 8 cols) ===")
display(avg_sd.iloc[:, :8].head())

for name in ["SLx_OUTPUTS_ROW1.xlsx", "SLx_OUTPUTS_ROW2.xlsx",
             "SLx_OUTPUTS_ROW3.xlsx", "SLx_OUTPUTS_ROW4.xlsx"]:
    df = pd.read_excel(out / name)
    print(f"{name}: {df.shape[0]} participants x {df.shape[1]} columns")

## 5. Visualize

A few example charts from the synthetic cohort: average bedtimes (weekday vs weekend nights), nightly sleep duration, and survey completion.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sleeplog.parsing import time_offset_from_noon

plt.rcParams["figure.figsize"] = (10, 4)

# --- Average bedtime: weekday vs weekend nights ---
bed_wk = avg_sd["SL_pre_BEDTIME_AVERAGE_WEEK_selfreport"]
bed_we = avg_sd["SL_pre_BEDTIME_AVERAGE_WEEKEND_selfreport"]
x = np.arange(len(avg_sd))
w = 0.35
fig, ax = plt.subplots()
ax.bar(x - w/2, [time_offset_from_noon(t) or np.nan for t in bed_wk], w, label="Weeknights")
ax.bar(x + w/2, [time_offset_from_noon(t) or np.nan for t in bed_we], w, label="Weekend nights")
ax.set_xticks(x, avg_sd["participant_number"].astype(int))
ax.set_ylabel("Hours after noon")
ax.set_xlabel("Participant")
ax.set_title("Average bedtime (in-bed time): weeknights vs weekend nights")
ax.legend()
plt.tight_layout()
plt.show()

# --- Nightly sleep duration distribution (from ROW1 daily variables) ---
row1 = pd.read_excel(out / "SLx_OUTPUTS_ROW1.xlsx")
durations = []
n_days = len(config.date_strings)
for d in range(1, n_days + 1):
    ib = row1[f"SL{d}_pre_BEDTIME_selfreport"].apply(time_offset_from_noon)
    wu = row1[f"SL{d}_pre_WOKETIME_selfreport"].apply(time_offset_from_noon)
    durations.extend((wu - ib).dropna().tolist())
durations = [v for v in durations if 0 < v < 16]

fig, ax = plt.subplots()
ax.hist(durations, bins=14, edgecolor="white")
ax.axvline(np.mean(durations), color="red", linestyle="--",
           label=f"Mean: {np.mean(durations):.1f} h")
ax.set_xlabel("Sleep duration (hours)")
ax.set_ylabel("Nights")
ax.set_title("Distribution of nightly sleep duration (synthetic cohort)")
ax.legend()
plt.tight_layout()
plt.show()

# --- Survey completion per participant ---
fig, ax = plt.subplots()
ax.bar(completion["participant_number"].astype(str),
       completion["SL_pre_weekdays_selfreport"], label="Weekdays")
ax.bar(completion["participant_number"].astype(str),
       completion["SL_pre_weekends_selfreport"],
       bottom=completion["SL_pre_weekdays_selfreport"], label="Weekends")
ax.set_xlabel("Participant")
ax.set_ylabel("Surveys completed")
ax.set_title("Survey completion per participant")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Download the outputs (Colab)

In [ ]:
try:
    from google.colab import files
    for name in ["AVERAGE_AND_SD_OUTPUTS.xlsx", "SL_OUTPUTS.xlsx",
                 "SLx_OUTPUTS_ROW1.xlsx", "SLx_OUTPUTS_ROW2.xlsx",
                 "SLx_OUTPUTS_ROW3.xlsx", "SLx_OUTPUTS_ROW4.xlsx"]:
        p = Path("outputs") / name
        if p.exists():
            files.download(str(p))
except ImportError:
    print("Not in Colab — outputs are in ./outputs/")